# A/B тест в мобильной игре Cookie Cats: где поставить ворота

**Задача.** В игре Cookie Cats на определённом уровне игрок упирается в ворота: нужно подождать или заплатить, чтобы продолжить. Разработчики провели A/B-тест,
чтобы выбрать, где их поставить:
- `gate_30` (контроль) — ворота на 30-м уровне
- `gate_40` (тест) — ворота на 40-м уровне

**Цель анализа:** определить, влияет ли перенос ворот на удержание игроков.

**Данные:** 90 189 игроков, установивших игру во время теста.

In [1]:
import pandas as pd
df = pd.read_csv('cookie_cats.csv')
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [2]:
df.shape

(90189, 5)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB


In [4]:
df['version'].value_counts()

version
gate_40    45489
gate_30    44700
Name: count, dtype: int64

In [5]:
df['userid'].nunique()

90189

In [6]:
from scipy import stats

observed = df['version'].value_counts().sort_index()  
n = len(df)
expected = [n / 2, n / 2]  

stats.chisquare(f_obs=observed, f_exp=expected)

Power_divergenceResult(statistic=6.9024049496058275, pvalue=0.008607987810836243)

In [7]:
df.groupby('version')['sum_gamerounds'].describe()

,count,mean,std,min,25%,50%,75%,max
version,,,,,,,,
gate_30,44700.0,52.456264,256.716423,0.0,5.0,17.0,50.0,49854.0
gate_40,45489.0,51.298776,103.294416,0.0,5.0,16.0,52.0,2640.0


In [8]:
df.sort_values('sum_gamerounds', ascending=False).head(10)

,userid,version,sum_gamerounds,retention_1,retention_7
57702,6390605,gate_30,49854,False,True
7912,871500,gate_30,2961,True,True
29417,3271615,gate_40,2640,True,False
43671,4832608,gate_30,2438,True,True
48188,5346171,gate_40,2294,True,True
46344,5133952,gate_30,2251,True,True
87007,9640085,gate_30,2156,True,True
36933,4090246,gate_40,2124,True,True
88328,9791599,gate_40,2063,True,True
6536,725080,gate_40,2015,True,True


In [9]:
df = df[df['sum_gamerounds'] < 49854]

In [10]:
df.shape

(90188, 5)

In [11]:
df.groupby('version')['sum_gamerounds'].describe()

,count,mean,std,min,25%,50%,75%,max
version,,,,,,,,
gate_30,44699.0,51.342111,102.057598,0.0,5.0,17.0,50.0,2961.0
gate_40,45489.0,51.298776,103.294416,0.0,5.0,16.0,52.0,2640.0


Распределение sum_gamerounds скошено вправо: среднее ~52 намного больше медианы ~17.
Обнаружен один аномальный игрок в группе gate_30 с 49854 раундами за 14 дней, это ~ 3560 раунов в день, он был удалён. После удаления стандартное отклонение в gte_30 снизилось с 257 до 102

In [12]:
df.groupby('version').agg( players = ('userid', 'count'), ret_1 = ('retention_1', 'mean'), ret_7 = ('retention_7', 'mean'))

,players,ret_1,ret_7
version,,,
gate_30,44699,0.448198,0.190183
gate_40,45489,0.442283,0.182000


In [13]:
from scipy import stats
import numpy as np

a = df[df['version'] == 'gate_30']['retention_1']  
b = df[df['version'] == 'gate_40']['retention_1']   

n_a, n_b = len(a), len(b)
x_a, x_b = a.sum(), b.sum()          
p_a, p_b = x_a / n_a, x_b / n_b   

p = (x_a + x_b) / (n_a + n_b)                        
se = np.sqrt(p * (1 - p) * ( 1/ n_a + 1 / n_b))
z = (p_b - p_a) / se

p_value = 2 * (1 - stats.norm.cdf(abs(z)))  
print(f'p_a={p_a:.4f}, p_b={p_b:.4f}, z={z:.3f}, p-value={p_value:.5f}')

p_a=0.4482, p_b=0.4423, z=-1.787, p-value=0.07392


In [14]:
def ztest_proportions(df, metric):
    a = df[df['version'] == 'gate_30'][metric]
    b = df[df['version'] == 'gate_40'][metric]

    n_a, n_b = len(a), len(b)
    x_a, x_b = a.sum(), b.sum()
    p_a, p_b = x_a / n_a, x_b / n_b
    diff = p_b - p_a

    p = (x_a + x_b) / (n_a + n_b)   
    se_test = np.sqrt(p * (1 - p) * ( 1/ n_a + 1 / n_b))
    z = diff / se_test
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))

    se_ci = np.sqrt(p_a*(1-p_a)/n_a + p_b * (1-p_b)/n_b)
    ci_low, ci_high = diff - 1.96 * se_ci, diff + 1.96 * se_ci

    print(f'{metric}: gate_30={p_a:.4f}, gate_40={p_b:.4f}, '
          f'разница={diff:.4f}, z={z:.3f}, p-value={p_value:.4f}, '
          f'95% ДИ=[{ci_low:.4f}; {ci_high:.4f}]')

ztest_proportions(df, 'retention_1')
ztest_proportions(df, 'retention_7')

retention_1: gate_30=0.4482, gate_40=0.4423, разница=-0.0059, z=-1.787, p-value=0.0739, 95% ДИ=[-0.0124; 0.0006]
retention_7: gate_30=0.1902, gate_40=0.1820, разница=-0.0082, z=-3.157, p-value=0.0016, 95% ДИ=[-0.0133; -0.0031]


Перенос ворот с 30 на 40 уровень не улучшает удержание, а даже снижает его примерно на 0.8 п.п.
Рекомендация: оставить ворота на 30-м уровне.
Возможно, ворота дают игроку паузу. Перерыв не даёт игре надоесть, и человек охотнее возвращается позже.

In [15]:
a = df[df['version'] == 'gate_30']['sum_gamerounds']
b = df[df['version'] == 'gate_40']['sum_gamerounds']

stats.mannwhitneyu(a, b, alternative='two-sided')

MannwhitneyuResult(statistic=1024285761.5, pvalue=0.05089155279145375)

$p_{value}$ > 0.05 

In [16]:
rng = np.random.default_rng(42)  
diffs = []

for _ in range(1000):
    sample_a = rng.choice(a, size=len(a), replace=True)
    sample_b = rng.choice(b, size = len(b), replace = True)
    diffs.append(np.median(sample_b) - np.median(sample_a))

ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
print(f'Разница медиан: 95% ДИ = [{ci_low}; {ci_high}]')

Разница медиан: 95% ДИ = [-1.0; 0.0]


Перенос ворот не меняет заметно, сколько игроки играют в первые две недели: медиана 17 раунов против 16. А retention 7-го дня падает. Значит, главный аргумент в решении - удержание.
Рекомендация: оставить ворота на 30 уровне

In [17]:
def mde_proportions(df, metric, alpha=0.05, power=0.8):
    a = df[df['version'] == 'gate_30'][metric]
    b = df[df['version'] == 'gate_40'][metric]
    n_a, n_b = len(a), len(b)
    p = a.mean()                             

    z_alpha = stats.norm.ppf(1 - alpha / 2)   
    z_power = stats.norm.ppf(power)           

    se = np.sqrt(p*(1-p)*(1/n_a + 1/n_b))
    mde = (z_alpha + z_power) * se

    print(f'{metric}: базовый уровень={p:.4f}, MDE={mde:.4f} '
          f'({mde * 100:.2f} п.п., относительно {mde / p * 100:.1f}%)')

mde_proportions(df, 'retention_1')
mde_proportions(df, 'retention_7')

retention_1: базовый уровень=0.4482, MDE=0.0093 (0.93 п.п., относительно 2.1%)
retention_7: базовый уровень=0.1902, MDE=0.0073 (0.73 п.п., относительно 3.9%)


Вывод "gate_40 снижает retention 7-го дня" надёжен, так как фактическая разница(0.82 п.п.) > MDE (0.73 п.п.)
А вот в retention_1 значимого эффекта не обнаружено, но тест недостаточно чувствителен, чтобы исключить эффект около 0.5 п.п.

## Итоговые выводы

**Корректность данных:** дублей и пропусков нет, каждый игрок встречается один раз. Обнаружен и удалён один аномальный
игрок с 49 854 раундами за 14 дней, вероятно, бот или
ошибка логирования.

**Результаты:**
| Метрика | gate_30 | gate_40 | Разница | Значимость |
|---|---|---|---|---|
| retention_1 | 44.82% | 44.23% | -0.59 п.п. | не значимо |
| retention_7 | 19.02% | 18.20% | -0.82 п.п. | значимо |
| раунды (медиана) | 17 | 16 | -1 | не значимо |

**Рекомендация:** оставить ворота на 30-м уровне

**Ограничения:** 
- Обнаружен SRM. Причину установить нельзя: в данных нет платформы, даты и версии
  приложения. Перекос небольшой, но в реальной работе тест стоило бы проверить
  до принятия решения.
- По retention_1 фактическая разница (0.59 п.п.) меньше MDE (0.93 п.п.), поэтому
  нельзя утверждать, что эффекта нет: тест недостаточно чувствителен для эффекта
  такого размера.